# 芯动联科（688582.SH）股票分析

本Notebook包含完整的数据分析和可视化代码。

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimSun', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('库导入成功！')

## 1. 读取数据

In [ ]:
# 读取股票数据
with open('stock_data.json', 'r', encoding='utf-8') as f:
    stock_data = json.load(f)

# 读取基本面数据
with open('daily_basic_data.json', 'r', encoding='utf-8') as f:
    basic_data = json.load(f)

# 转换为DataFrame
df_stock = pd.DataFrame(stock_data)
df_basic = pd.DataFrame(basic_data)

print(f'股票数据条数: {len(df_stock)}')
print(f'基本面数据条数: {len(df_basic)}')

## 2. 技术指标计算

In [ ]:
# 数据处理
df_stock['trade_date'] = pd.to_datetime(df_stock['trade_date'], format='%Y%m%d')
df_basic['trade_date'] = pd.to_datetime(df_basic['trade_date'], format='%Y%m%d')

df_stock = df_stock.sort_values('trade_date').reset_index(drop=True)
df_basic = df_basic.sort_values('trade_date').reset_index(drop=True)

# 合并数据
df = pd.merge(df_stock, df_basic, on='trade_date', how='left')

# 计算移动平均线
df['MA5'] = df['close'].rolling(window=5).mean()
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()
df['MA60'] = df['close'].rolling(window=60).mean()

# 计算MACD
exp1 = df['close'].ewm(span=12, adjust=False).mean()
exp2 = df['close'].ewm(span=26, adjust=False).mean()
df['MACD_DIF'] = exp1 - exp2
df['MACD_DEA'] = df['MACD_DIF'].ewm(span=9, adjust=False).mean()
df['MACD_HIST'] = (df['MACD_DIF'] - df['MACD_DEA']) * 2

print('技术指标计算完成！')
print(df[['trade_date', 'close', 'MA5', 'MA10', 'MA20', 'MACD_DIF']].tail())

## 3. 可视化分析

In [ ]:
# 绘制K线图和均线
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 8))

# 绘制K线
for i in range(len(df)):
    date = df.iloc[i]['trade_date']
    open_price = df.iloc[i]['open']
    close_price = df.iloc[i]['close']
    high_price = df.iloc[i]['high']
    low_price = df.iloc[i]['low']
    
    color = 'red' if close_price >= open_price else 'green'
    ax.plot([date, date], [low_price, high_price], color=color, linewidth=1, alpha=0.8)
    ax.plot([date, date], [open_price, close_price], color=color, linewidth=4, alpha=0.8)

# 绘制均线
ax.plot(df['trade_date'], df['MA5'], label='MA5', linewidth=1.5)
ax.plot(df['trade_date'], df['MA10'], label='MA10', linewidth=1.5)
ax.plot(df['trade_date'], df['MA20'], label='MA20', linewidth=1.5)
ax.plot(df['trade_date'], df['MA60'], label='MA60', linewidth=1.5)

ax.set_title('芯动联科（688582.SH）- K线图与均线分析', fontsize=16, fontweight='bold')
ax.set_xlabel('日期')
ax.set_ylabel('价格（元）')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 绘制MACD指标
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[2, 1])

ax1.plot(df['trade_date'], df['close'], color='black', linewidth=2, label='收盘价')
ax1.set_title('MACD指标分析', fontsize=16, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df['trade_date'], df['MACD_DIF'], label='DIF', linewidth=1.5)
ax2.plot(df['trade_date'], df['MACD_DEA'], label='DEA', linewidth=1.5)
colors = ['red' if x > 0 else 'green' for x in df['MACD_HIST']]
ax2.bar(df['trade_date'], df['MACD_HIST'], color=colors, alpha=0.5, label='MACD')
ax2.set_xlabel('日期')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. 数据分析结论

In [ ]:
# 输出关键指标
print('=== 芯动联科（688582.SH）关键指标 ===')
print(f'最新收盘价: {df["close"].iloc[-1]:.2f}元')
print(f'最新PE: {df["pe"].iloc[-1]:.2f}')
print(f'最新PB: {df["pb"].iloc[-1]:.2f}')
print(f'最新PS: {df["ps"].iloc[-1]:.2f}')
print(f'总市值: {df["total_mv"].iloc[-1]/10000:.2f}亿元')
print(f'换手率: {df["turnover_rate"].iloc[-1]:.2f}%')
print(f'量比: {df["volume_ratio"].iloc[-1]:.2f}')